In [5]:
stores["store_id"] = (
    stores["store_id"]
    .astype(str)
    .str.strip()
)

demographics["store_id"] = (
    demographics["store_id"]
    .astype(str)
    .str.strip()
)

In [6]:
weather["date"] = pd.to_datetime(
    weather["date"]
).dt.normalize()

In [7]:
if "snapshot_date" in promotions.columns:

    promotions["snapshot_date"] = pd.to_datetime(
        promotions["snapshot_date"]
    ).dt.normalize()

elif "flyer_week_start" in promotions.columns:

    promotions["snapshot_date"] = pd.to_datetime(
        promotions["flyer_week_start"]
    ).dt.normalize()

else:
    raise ValueError(
        "No promotion date column found."
    )

In [8]:
# Standardize promotion columns

if "promo_price" not in promotions.columns:

    if "current_price" in promotions.columns:

        promotions["promo_price"] = (
            promotions["current_price"]
        )

    else:
        promotions["promo_price"] = np.nan

In [9]:
if "regular_price" not in promotions.columns:
    promotions["regular_price"] = np.nan

In [10]:
if "discount_amount" not in promotions.columns:

    promotions["discount_amount"] = (
        promotions["regular_price"]
        - promotions["promo_price"]
    )

In [11]:
if "discount_pct" not in promotions.columns:

    promotions["discount_pct"] = np.where(
        promotions["regular_price"] > 0,

        (
            promotions["discount_amount"]
            / promotions["regular_price"]
        ) * 100,

        np.nan
    )

# Merge Walmart + OSM + demographics

In [12]:
print(
    "Store context IDs:",
    stores["store_id"].nunique()
)

print(
    "Demographic IDs:",
    demographics["store_id"].nunique()
)

Store context IDs: 15
Demographic IDs: 15


In [13]:
store_master = stores.merge(
    demographics,
    on="store_id",
    how="left",
    suffixes=("", "_demo")
)

print(store_master.shape)

display(store_master.head())

(15, 24)


,store_id,store_name,address,city,province,postal_code,full_address,latitude,longitude,universities,...,commercial,retail_landuse,rail_stations,store_name_demo,population_5km,population_density_5km,median_household_income_5km,average_household_size_5km,employment_rate_5km,n_census_tracts
0,3106,Toronto Dufferin Mall,900 Dufferin St,Toronto,ON,M6H 4A9,"900 Dufferin St, Toronto, ON M6H 4A9, Canada",43.655980,-79.435728,19,...,172,519,44,Toronto Dufferin Mall,593613.628819,15197.462826,86252.286112,2.017781,62.389316,147
1,1139,Toronto Downsview,3757 Keele St,Toronto,ON,M3J 1N4,"3757 Keele St, Toronto, ON M3J 1N4, Canada",43.757509,-79.489316,3,...,82,216,18,Toronto Downsview,237963.886322,5979.220710,80848.194959,2.764884,51.253500,63
2,1004,Toronto Stockyards,2525 St Clair Ave W,Toronto,ON,M6N 4Z5,"2525 St Clair Ave W, Toronto, ON M6N 4Z5, Canada",43.667987,-79.484753,0,...,145,498,24,Toronto Stockyards,423374.384031,8590.969933,92072.905741,2.383569,56.936454,123
3,1150,Toronto Downtown East,1000 Gerrard St E,Toronto,ON,M4M 3G6,"1000 Gerrard St E, Toronto, ON M4M 3G6, Canada",43.669300,-79.339861,20,...,108,424,36,Toronto Downtown East,482564.923619,16864.257792,88944.349902,2.079208,59.227363,118
4,3105,Toronto North Park,1305 Lawrence Ave W,Toronto,ON,M6L 1A5,"1305 Lawrence Ave W, Toronto, ON M6L 1A5, Canada",43.708480,-79.473550,1,...,128,427,17,Toronto North Park,373912.266144,6242.331511,85562.143106,2.569160,53.592422,100


# Check demographic coverage

In [14]:
demo_cols = [
    col for col in [
        "population_5km",
        "population_density_5km",
        "median_household_income_5km",
        "average_household_size_5km",
        "employment_rate_5km"
    ]
    if col in store_master.columns
]

display(
    store_master[
        demo_cols
    ]
    .isna()
    .sum()
    .to_frame("missing")
)

,missing
population_5km,0
population_density_5km,0
median_household_income_5km,0
average_household_size_5km,0
employment_rate_5km,0


# Promotions are regional

This is important.

Our flyer data is not proven to be unique for each Walmart store.

So for V1:

each Ontario/Canada flyer product
        ↓
candidate promotion for every GTA store


We are not claiming every store has identical inventory.


# Cross join stores × promotions

In [15]:
store_master["_join_key"] = 1
promotions["_join_key"] = 1

context = store_master.merge(
    promotions,
    on="_join_key",
    how="inner",
    suffixes=("_store", "_promo")
)

context = context.drop(
    columns="_join_key"
)

stores.drop(
    columns="_join_key",
    inplace=True,
    errors="ignore"
)

promotions.drop(
    columns="_join_key",
    inplace=True,
    errors="ignore"
)

print(context.shape)

(105, 44)


In [16]:
expected = (
    len(store_master)
    * len(promotions)
)

print("Expected:", expected)
print("Actual:", len(context))

Expected: 105
Actual: 105


# Add weather

In [17]:
weather_cols = [
    "date",
    "mean_temperature",
    "min_temperature",
    "max_temperature",
    "total_rain",
    "total_snow",
    "total_precipitation",
    "is_rain",
    "is_snow"
]

weather_cols = [
    c for c in weather_cols
    if c in weather.columns
]

weather_small = (
    weather[weather_cols]
    .drop_duplicates("date")
    .sort_values("date")
)

In [18]:
context = context.merge(
    weather_small,
    left_on="snapshot_date",
    right_on="date",
    how="left"
)

context = context.drop(
    columns="date",
    errors="ignore"
)

In [19]:
print(
    "Missing weather:",
    context["mean_temperature"]
    .isna()
    .mean()
    * 100,
    "%"
)

Missing weather: 100.0 %


In [25]:
# Sort both datasets
context = context.sort_values("snapshot_date").reset_index(drop=True)

weather_small = (
    weather_small
    .sort_values("date")
    .reset_index(drop=True)
)

# Remove weather columns if already created by previous failed merge
for col in weather_small.columns:
    if col != "date":
        context.drop(
            columns=col,
            inplace=True,
            errors="ignore"
        )

# Join each promotion snapshot to the latest previous weather observation
context = pd.merge_asof(
    context,
    weather_small,
    left_on="snapshot_date",
    right_on="date",
    direction="backward",
    tolerance=pd.Timedelta("3D")
)

context = context.rename(
    columns={"date": "weather_date"}
)

In [26]:
print(
    "Missing weather:",
    round(
        context["mean_temperature"]
        .isna()
        .mean() * 100,
        2
    ),
    "%"
)

print(
    context[
        [
            "snapshot_date",
            "weather_date",
            "mean_temperature",
            "total_rain",
            "total_snow"
        ]
    ]
    .drop_duplicates()
    .head(20)
)

Missing weather: 100.0 %
  snapshot_date weather_date  mean_temperature  total_rain  total_snow
0    2026-09-11   2026-09-11               NaN         NaN         NaN


In [22]:
for col in weather_cols:

    if col == "date":
        continue

    if col in fallback.columns:

        context.loc[
            fallback["index"],
            col
        ] = fallback[col].values

# Calendar features

In [23]:
context["year"] = (
    context["snapshot_date"].dt.year
)

context["month"] = (
    context["snapshot_date"].dt.month
)

context["day_of_week"] = (
    context["snapshot_date"].dt.dayofweek
)

context["week_of_year"] = (
    context["snapshot_date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

context["is_weekend"] = (
    context["day_of_week"]
    .isin([5, 6])
    .astype("int8")
)

# Product/promotion quality checks

In [27]:
print(context.columns.tolist())

['store_id_store', 'store_name', 'address', 'city', 'province', 'postal_code', 'full_address', 'latitude', 'longitude', 'universities', 'colleges', 'schools', 'offices', 'industrial', 'commercial', 'retail_landuse', 'rail_stations', 'store_name_demo', 'population_5km', 'population_density_5km', 'median_household_income_5km', 'average_household_size_5km', 'employment_rate_5km', 'n_census_tracts', 'product_id', 'product_name', 'category', 'current_price', 'regular_price', 'savings', 'promotion_type', 'product_url', 'page_number', 'snapshot_date', 'country', 'region', 'store_id_promo', 'source', 'captured_at', 'discount_amount', 'discount_pct', 'is_promoted', 'promo_id', 'promo_price', 'year', 'month', 'day_of_week', 'week_of_year', 'is_weekend', 'weather_date', 'mean_temperature', 'min_temperature', 'max_temperature', 'total_rain', 'total_snow', 'total_precipitation', 'is_rain', 'is_snow']


In [28]:
promotions_for_join = promotions.drop(
    columns=["store_id"],
    errors="ignore"
).copy()

store_master["_join_key"] = 1
promotions_for_join["_join_key"] = 1

context = store_master.merge(
    promotions_for_join,
    on="_join_key",
    how="inner"
)

context.drop(
    columns="_join_key",
    inplace=True
)

# Discount checks

In [29]:
print(
    "Negative prices:",
    (context["promo_price"] < 0).sum()
)

print(
    "Discount < 0:",
    (context["discount_pct"] < 0).sum()
)

print(
    "Discount > 100:",
    (context["discount_pct"] > 100).sum()
)

Negative prices: 0
Discount < 0: 0
Discount > 100: 0


# Create contextual features

In [30]:
if "mean_temperature" in context.columns:

    context["is_hot"] = (
        context["mean_temperature"] >= 25
    ).astype("int8")

    context["is_cold"] = (
        context["mean_temperature"] <= 0
    ).astype("int8")

In [31]:
if "total_snow" in context.columns:

    context["snow_day"] = (
        context["total_snow"]
        .fillna(0) > 0
    ).astype("int8")

In [32]:
if "total_rain" in context.columns:

    context["rain_day"] = (
        context["total_rain"]
        .fillna(0) > 0
    ).astype("int8")

# Unique recommendation-row ID

In [33]:
def create_context_id(row):

    product_key = (
        row.get("product_id")
        if pd.notna(
            row.get("product_id")
        )
        else row.get("product_name")
    )

    raw = (
        f"{row['store_id']}|"
        f"{product_key}|"
        f"{row['snapshot_date']}"
    )

    return hashlib.sha1(
        raw.encode("utf-8")
    ).hexdigest()[:20]

In [34]:
context["context_id"] = (
    context.apply(
        create_context_id,
        axis=1
    )
)

In [35]:
print(
    "Duplicate context IDs:",
    context["context_id"]
    .duplicated()
    .sum()
)

Duplicate context IDs: 0


# Inspect final dataset

In [36]:
important_cols = [
    c for c in [
        "context_id",
        "store_id",
        "store_name",
        "city",

        "product_id",
        "product_name",
        "category",

        "regular_price",
        "promo_price",
        "discount_pct",
        "is_promoted",

        "population_5km",
        "population_density_5km",
        "median_household_income_5km",
        "employment_rate_5km",

        "universities",
        "colleges",
        "offices",
        "industrial",
        "commercial",

        "mean_temperature",
        "total_rain",
        "total_snow",

        "month",
        "day_of_week",
        "is_weekend"
    ]
    if c in context.columns
]

display(
    context[
        important_cols
    ].head(20)
)

,context_id,store_id,store_name,city,product_id,product_name,category,regular_price,promo_price,discount_pct,is_promoted,population_5km,population_density_5km,median_household_income_5km,employment_rate_5km,universities,colleges,offices,industrial,commercial
0,5c8f6edd755e34fde827,3106,Toronto Dufferin Mall,Toronto,10142121,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,NaN,97.00,NaN,1,593613.628819,15197.462826,86252.286112,62.389316,19,19,940,109,172
1,80eb63cb821d22687b8c,3106,Toronto Dufferin Mall,Toronto,10316379,"EGGO Blueberry Flavour Waffles, 280g (8 waffle...",Grocery,3.97,2.97,25.188917,1,593613.628819,15197.462826,86252.286112,62.389316,19,19,940,109,172
2,bd79ba49ce5c202840cb,3106,Toronto Dufferin Mall,Toronto,6000202703709,Quaker Chewy Chocolate Chips Granola Bars Vari...,Grocery,13.44,10.97,18.377976,1,593613.628819,15197.462826,86252.286112,62.389316,19,19,940,109,172
3,cae9459736ebe7205a48,3106,Toronto Dufferin Mall,Toronto,6000199413869,Tim Hortons Original Blend Medium Roast Coffee...,Grocery,25.47,19.86,22.025913,1,593613.628819,15197.462826,86252.286112,62.389316,19,19,940,109,172
4,33aa90af8aba68777349,3106,Toronto Dufferin Mall,Toronto,6000199788886,"Janes® Pub Style Chicken Strips, 700g, Frozen ...",Grocery,9.97,5.98,40.020060,1,593613.628819,15197.462826,86252.286112,62.389316,19,19,940,109,172
5,8e68d32d5d6355a4e71f,3106,Toronto Dufferin Mall,Toronto,6000199044869,Sealtest Chocolate Partly Skimmed 1% Milk 750ml,Grocery,NaN,1.93,NaN,0,593613.628819,15197.462826,86252.286112,62.389316,19,19,940,109,172
6,92a9e2ac7cf849562d5d,3106,Toronto Dufferin Mall,Toronto,6000188742528,"Red Bull Energy Drink, 250 ml (4 pack)",Grocery,NaN,10.97,NaN,0,593613.628819,15197.462826,86252.286112,62.389316,19,19,940,109,172
7,4ddea764dfd13b78ede5,1139,Toronto Downsview,Toronto,10142121,"Campbell's® Condensed Cream of Mushroom Soup, ...",Grocery,NaN,97.00,NaN,1,237963.886322,5979.220710,80848.194959,51.253500,3,8,245,159,82
8,86190d72d75d79148d1d,1139,Toronto Downsview,Toronto,10316379,"EGGO Blueberry Flavour Waffles, 280g (8 waffle...",Grocery,3.97,2.97,25.188917,1,237963.886322,5979.220710,80848.194959,51.253500,3,8,245,159,82
9,cf1c1f48373667d0cefc,1139,Toronto Downsview,Toronto,6000202703709,Quaker Chewy Chocolate Chips Granola Bars Vari...,Grocery,13.44,10.97,18.377976,1,237963.886322,5979.220710,80848.194959,51.253500,3,8,245,159,82


# Missingness report

In [37]:
missing_report = (
    context
    .isna()
    .mean()
    .mul(100)
    .sort_values(
        ascending=False
    )
    .to_frame("missing_pct")
)

display(
    missing_report.head(30)
)

,missing_pct
region,100.000000
promotion_type,85.714286
discount_pct,42.857143
regular_price,42.857143
discount_amount,28.571429
savings,28.571429
latitude,6.666667
longitude,6.666667
page_number,0.000000
product_name,0.000000


# Save

In [38]:
context.to_parquet(
    OUTPUT_PATH,
    index=False
)

print(
    "Saved:",
    OUTPUT_PATH
)

print(
    "Rows:",
    len(context)
)

print(
    "Size MB:",
    round(
        OUTPUT_PATH.stat().st_size
        / 1024**2,
        2
    )
)

Saved: data/processed/gta_promotion_context_v1.parquet
Rows: 105
Size MB: 0.03


In [ ]:
# # What 12 accomplished

# <!-- Before:

# Walmart locations     separate
# OSM                   separate
# StatsCan              separate
# Weather               separate
# Promotions            separate

# After:

# STORE × PRODUCT × FLYER DATE

# Store:
#   location
#   neighborhood POIs
#   demographics

# Product:
#   name
#   category
#   price
#   discount

# Context:
#   weather
#   calendar
#   promotion -->